# Imaging right: take a sample to the right imaging rack

The right imaging rack stands behind the sample racks. A sample on it lies left to right with the metal
end on the left and the coated end on the right, under the camera. The spot (slot 2 from the rack's front
screw) was found with `imaging_quick_check_right.ipynb`.

**Cells, in order:**

1. **Connect** to the arm.
2. **Settings**: which sample, dip speeds, the imaging spot and the travel speeds. Nothing moves here;
   run it before any of the cells below, and again after changing a setting.
3. **Full run**: pick up the sample, dip, hold and lift exactly as the immersion sweep in `AIsorb_arm.ipynb`
   does (through `mid_station`), then leave it on the imaging rack instead of its own slot.
4. **Test, no dipping**: take the sample from its own slot straight to the imaging rack. Use a dry sample.
5. **Teach the put-down spot**: take a dry sample to the imaging rack and hold it just above the slot, so you
   can move it with typed commands until it sits exactly right; then let go and send Claude the spot.
6. **Take back**: fetch the sample from the imaging rack and put it back in its own slot.
7. **Only after answering no** at the stop above the imaging rack: put the held sample back in its own slot.

**How the sample goes onto the imaging rack.** So that the electrode never comes down onto the camera,
the arm does not come straight down onto the spot. It crosses high above the bench at 100 mm/s, turning the
gripper a quarter turn, comes down 7 cm to the left of the spot (away from the camera) and stops there to
ask. It then lowers the electrode to 2.5 cm above the rack, slides it right under the camera at 30 mm/s,
puts it down at 10 mm/s, lets go, lifts 2.5 cm and slides back out. Taking it back is the same, in reverse.

**Before running:** the camera lens is at least 5 cm above the rack, the red wire on the gripper has slack,
and your hand is on the e-stop. Do not run anything in another notebook that is connected to the arm at
the same time. The first times, leave `CONFIRM_BEFORE_PLACING = True`.

In [ ]:
from robotic_testing.platforms import init_vla_platform

rt = init_vla_platform(exp_name='imaging_right', connect_db=False)
arm = rt.arm

In [ ]:
# ===== Settings and moves for the right imaging rack. Run this cell first; nothing moves here. =====
import time

SAMPLE = 0                    # which sample (its usual rack slot)
DIP_SPEED_MM_S = 20.0         # speed going into the liquid
HOLD_S = 1                    # seconds to stay immersed
LIFT_SPEED_MM_S = 20.0        # speed coming out of the liquid

# Right imaging rack, slot 2 from the front screw (metal end on the left, coated end on the right,
# towards the camera). The quick check found x = 294, y = 488.1; a sample put down there landed a little
# off, so x is 2 mm further right. The gripper lets go of the electrode here. z = 190.6 is the height the
# arm puts samples down at in the sample racks. If the sample lands too far left or right, change x (the
# first number): bigger = further right, in mm. Then run this cell again.
IMAGING_SPOT = [296.0, 488.1, 190.6, 180.0, 0.0, 180.0]
SAFE_Z = 500.0                # travel height between mid_station and the imaging rack
HOVER_MM = 100.0              # where the arm stops to ask, above the spot beside the imaging spot
SLIDE_MM = 70.0               # how far towards the metal end (left) the arm comes down, away from the camera
SLIDE_HEIGHT_MM = 25.0        # height above the imaging spot at which the electrode slides in and out

# Speeds after the dip (mm/s, at most 100), so the film has as little time as possible to dry on the way:
TRAVEL_SPEED_MM_S = 100.0     # from mid_station: up, across, and down beside the imaging rack
SLIDE_SPEED_MM_S = 30.0       # down low over the rack, and sliding in under the camera
PLACE_SPEED_MM_S = 10.0       # the last 2.5 cm down into the slot
CONFIRM_BEFORE_PLACING = True # stop above the imaging rack and wait for your yes (the wait adds drying
                              # time: set False once a run has gone well)

cfg = arm.config


def at_speed(mm_s):
    """Default motion settings with the speed swapped to mm_s."""
    if not 0 < mm_s <= 100:
        raise ValueError(f'Speed {mm_s} mm/s must be between 0 and 100')
    return {**cfg.pos_settings_dict['default'], 'speed': mm_s}


def turned_from(rpy_a, rpy_b):
    """Largest angle (degrees) between two orientations given as roll, pitch, yaw."""
    return max(abs(((a - b) + 180) % 360 - 180) for a, b in zip(rpy_a, rpy_b))


def go(steps):
    for name, pose, speed in steps:
        print('   ->', name)
        arm.move_to_pos(pose, **at_speed(speed))


for speed in (DIP_SPEED_MM_S, LIFT_SPEED_MM_S, TRAVEL_SPEED_MM_S, SLIDE_SPEED_MM_S, PLACE_SPEED_MM_S):
    at_speed(speed)           # raises now if a speed is out of range, before anything moves

mid = list(cfg.pos_dict['mid_station'])
immersed = cfg.pos_dict['flask_contact_immersed']
above_flask = list(immersed[:2]) + [immersed[2] + cfg.hover_offset_dict['flask']] + list(immersed[3:])
img_rpy = IMAGING_SPOT[3:]


def routes_for(x, y, z):
    """The moves to and from an imaging spot at x, y, z (z = the height the gripper lets go at)."""
    sx, sz = x - SLIDE_MM, z + SLIDE_HEIGHT_MM   # beside the spot, away from the camera; slide height
    to_rack = [   # mid_station -> high across, turning -> down beside the imaging spot (where the arm stops to ask)
        ('straight up from mid_station', [mid[0], mid[1], SAFE_Z, *mid[3:]], TRAVEL_SPEED_MM_S),
        ('across, turning a quarter turn', [sx, y, SAFE_Z, *img_rpy], TRAVEL_SPEED_MM_S),
        ('down, beside the imaging spot', [sx, y, z + HOVER_MM, *img_rpy], TRAVEL_SPEED_MM_S),
    ]
    into_slot = [   # beside the imaging spot -> low over the rack -> slide in under the camera -> down into the slot
        ('down, low over the rack', [sx, y, sz, *img_rpy], SLIDE_SPEED_MM_S),
        ('slide towards the camera, to the imaging spot', [x, y, sz, *img_rpy], SLIDE_SPEED_MM_S),
        ('down into the slot', [x, y, z, *img_rpy], PLACE_SPEED_MM_S),
    ]
    out_to_mid = [   # up a little -> slide out, away from the camera -> high across, turning back -> mid_station
        ('up a little', [x, y, sz, *img_rpy], PLACE_SPEED_MM_S),
        ('slide back, away from the camera', [sx, y, sz, *img_rpy], SLIDE_SPEED_MM_S),
        ('straight up', [sx, y, SAFE_Z, *img_rpy], TRAVEL_SPEED_MM_S),
        ('across, turning back', [mid[0], mid[1], SAFE_Z, *mid[3:]], TRAVEL_SPEED_MM_S),
        ('down to mid_station', mid, TRAVEL_SPEED_MM_S),
    ]
    return to_rack, into_slot, out_to_mid


def use_spot(x, y, z):
    """Make the cells below use the imaging spot x, y, z (until this settings cell is run again)."""
    global img_x, img_y, img_z, side_x, slide_z, to_imaging, slide_in, slide_out_to_mid
    img_x, img_y, img_z = x, y, z
    side_x, slide_z = x - SLIDE_MM, z + SLIDE_HEIGHT_MM
    to_imaging, slide_in, slide_out_to_mid = routes_for(x, y, z)


use_spot(*IMAGING_SPOT[:3])


def route_to_mid_from_here():
    """
    Moves that bring the arm to mid_station high above the bench, or [] if it is already there.
    Stops the cell (nothing moved) unless the gripper points straight down at least 5 cm above the racks.
    """
    here = arm.get_cartesian_pos()
    if min(turned_from(here[3:], mid[3:]), turned_from(here[3:], img_rpy)) > 5 or here[2] < 240:
        raise SystemExit('The gripper must point straight down and be at least 5 cm above the racks (z >= 240) '
                         'before this cell. The arm did not move.')
    if max(abs(a - b) for a, b in zip(here[:3], mid[:3])) <= 5 and turned_from(here[3:], mid[3:]) <= 5:
        return []
    return [
        ('straight up from where the arm is', [here[0], here[1], SAFE_Z, *here[3:]], 30),
        ('across to above mid_station', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
        ('down to mid_station', mid, 30),
    ]


def ask_to_start(what, route_to_mid):
    print(what)
    for name, pose, speed in route_to_mid:
        print(f'  first: {name:40s} -> {[round(v, 1) for v in pose]} at {speed} mm/s')
    print('Check: red wire has slack, hand on the e-stop, and the camera lens at least 5 cm above the rack.')
    if input('Type yes to start: ').strip().lower() != 'yes':
        raise SystemExit('Cancelled, the arm did not move.')
    arm.assert_ready()
    if route_to_mid:
        print('[0] to mid_station, high above the bench')
        go(route_to_mid)


def leave_on_imaging_rack():
    """From mid_station, holding the sample: put it in the imaging slot, let go, back to mid_station."""
    print(f'[*] to the right imaging rack at {TRAVEL_SPEED_MM_S} mm/s')
    go(to_imaging)
    if CONFIRM_BEFORE_PLACING and input('Gripper above the imaging rack, coated end on the right, nothing in '
                                        'the way of sliding in under the camera? Type yes to put the sample down: '
                                        ).strip().lower() != 'yes':
        raise SystemExit('Stopped above the imaging rack, still holding the sample. '
                         'Run the last cell to put it back in its own slot.')
    go(slide_in)
    arm.open_gripper()
    print('[*] sample left on the imaging rack; back to mid_station')
    go(slide_out_to_mid)


print(f'Settings loaded: sample {SAMPLE}, imaging spot {[round(v, 1) for v in IMAGING_SPOT]}, '
      f'travel {TRAVEL_SPEED_MM_S} mm/s after the dip. Nothing has moved.')

### Full run: dip, then leave the sample on the imaging rack

In [ ]:
# ===== Dip the sample, then leave it on the right imaging rack (not back in its own slot) =====
# Pick up -> mid_station -> above the flask -> dip -> hold -> lift -> mid_station   (as in AIsorb_arm.ipynb)
# -> high across, turning -> down beside the imaging spot -> slide in under the camera -> into the slot
# -> let go -> slide back out -> high across, turning back -> mid_station.
route = route_to_mid_from_here()
ask_to_start(f'Sample {SAMPLE}: dip {DIP_SPEED_MM_S} mm/s, hold {HOLD_S} s, lift {LIFT_SPEED_MM_S} mm/s, '
             f'then leave it on the right imaging rack. Imaging slot must be empty.', route)

print(f'[1] pick up sample {SAMPLE}, then to the flask')
arm.pick_up_sample(SAMPLE)
arm.move_to_mid_station()
arm.move_to_flask()

print(f'[2] dip at {DIP_SPEED_MM_S} mm/s')
arm.move_to_pos(immersed, **at_speed(DIP_SPEED_MM_S))
arm.sample_in_flask = True
if HOLD_S > 0:
    print(f'[3] hold {HOLD_S} s')
    time.sleep(HOLD_S)

print(f'[4] lift at {LIFT_SPEED_MM_S} mm/s, then to mid_station (at the arm\'s usual 100 mm/s)')
arm.move_to_pos(above_flask, **at_speed(LIFT_SPEED_MM_S))
arm.sample_in_flask = False
arm.move_to_mid_station()

leave_on_imaging_rack()
print(f'Done. Sample {SAMPLE} is on the right imaging rack. Its own slot is empty.')

### Test, no dipping: from its own slot to the imaging rack

Use a dry sample. Afterwards, the next cell takes it back.

In [ ]:
# ===== TEST, no dipping: take the sample from its own slot straight to the right imaging rack =====
# Pick up -> mid_station -> high across, turning -> down beside the imaging spot -> slide in under the camera
# -> into the slot -> let go -> slide back out -> high across, turning back -> mid_station.
# Use a dry sample: nothing goes into the liquid.
route = route_to_mid_from_here()
ask_to_start(f'TEST: sample {SAMPLE} from its own slot to the right imaging rack, no dipping. '
             f'Imaging slot must be empty.', route)

print(f'[1] pick up sample {SAMPLE}, then to mid_station')
arm.pick_up_sample(SAMPLE)
arm.move_to_mid_station()

leave_on_imaging_rack()
print(f'Done. Sample {SAMPLE} is on the right imaging rack. Run the next cell to take it back to its own slot.')

### Teach the put-down spot with a real sample

Use a dry sample. The arm holds it 5 mm above the slot; move it with `l 2`, `r 2`, `f 1`, `b 1`, `d 1`, `u 1` until it sits exactly where you want, then type `release`. Send Claude the `Spot:` line it prints. Afterwards the take-back cell fetches the sample from that spot (until the settings cell is run again).

In [ ]:
# ===== TEACH the put-down spot with a real sample (no dipping) =====
# Pick up -> mid_station -> high across, turning -> beside the imaging spot (asks) -> slide in under the camera
# -> down to 5 mm above the slot, STILL HOLDING the sample. Then you move it with typed commands until it is
# exactly where you want it, and type release to let go. Use a dry sample; the imaging slot must be empty.
# Commands (directions as seen standing in front of the bench, facing the wall):
#   l 2 / r 2   -> 2 mm left / right                     f 2 / b 2 -> 2 mm towards you / towards the wall
#   d 1 / u 1   -> 1 mm lower / higher
#   release     -> let go here (asks first), print the spot for Claude, then move the arm away
#   back        -> give up: lift, slide out, and put the sample back in its own slot
# Sideways steps are at most 5 mm, height steps at most 3 mm. The sample stays within 30 mm sideways of the
# planned spot, and between 5 mm below and 25 mm above the usual put-down height. Before moving towards you
# or towards the wall, go up a little (u 3) so the electrode clears the teeth of the rack.

MOVES = {'f': (0, -1), 'b': (0, 1), 'l': (-1, 0), 'r': (1, 0)}
plan_x, plan_y, plan_z = img_x, img_y, img_z


def where_text(x, y, z):
    """Where the sample is now, compared with the planned spot, in words."""
    parts = []
    if abs(x - plan_x) >= 0.05:
        parts.append(f'{abs(x - plan_x):.1f} mm to the ' + ('right' if x > plan_x else 'left'))
    if abs(y - plan_y) >= 0.05:
        parts.append(f'{abs(y - plan_y):.1f} mm towards ' + ('the wall' if y > plan_y else 'you'))
    if abs(z - plan_z) >= 0.05:
        parts.append(f'{abs(z - plan_z):.1f} mm ' + ('above' if z > plan_z else 'below') + ' the put-down height')
    return ', '.join(parts) if parts else 'exactly on the planned spot'


route = route_to_mid_from_here()
ask_to_start(f'TEACH: sample {SAMPLE} from its own slot to the right imaging rack, held there for you to '
             f'fine-tune, no dipping. Imaging slot must be empty.', route)

print(f'[1] pick up sample {SAMPLE}, then to mid_station')
arm.pick_up_sample(SAMPLE)
arm.move_to_mid_station()
print('[2] to the right imaging rack')
go(to_imaging)
if input('Gripper above the imaging rack, coated end on the right, nothing in the way of sliding in under the '
         'camera? Type yes to slide the sample in: ').strip().lower() != 'yes':
    raise SystemExit('Stopped above the imaging rack, still holding the sample. '
                     'Run the last cell to put it back in its own slot.')
go(slide_in[:2])                          # low over the rack, then slide in under the camera
tx, ty, tz = plan_x, plan_y, plan_z + 5
arm.move_to_pos([tx, ty, tz, *img_rpy], **at_speed(PLACE_SPEED_MM_S))
print('[3] holding the sample 5 mm above the slot. Move it until it is where you want it, then type release.')

released = False
while True:
    cmd = input('l/r/f/b or d/u + mm, or release, back: ').strip().lower().split()
    if not cmd:
        continue
    if cmd == ['release']:
        if input('Sample exactly where you want it? Type yes to let go: ').strip().lower() == 'yes':
            arm.open_gripper()
            released = True
            break
        continue
    if cmd == ['back']:
        break
    if len(cmd) != 2 or cmd[0] not in ('l', 'r', 'f', 'b', 'd', 'u'):
        print('Did not understand. Examples: r 2, l 1, f 1, b 1, d 1, u 3, release, back.')
        continue
    try:
        mm = float(cmd[1])
    except ValueError:
        print('The second part must be a number of mm, e.g. r 2.')
        continue
    nx, ny, nz = tx, ty, tz
    if cmd[0] in ('d', 'u'):
        if not 0 < mm <= 3:
            print('Use a distance between 0 and 3 mm.')
            continue
        nz = tz - mm if cmd[0] == 'd' else tz + mm
        if not plan_z - 5 <= nz <= plan_z + SLIDE_HEIGHT_MM:
            print('That is outside the allowed height, so the arm did not move. Tell Claude.')
            continue
    else:
        if not 0 < mm <= 5:
            print('Use a distance between 0 and 5 mm.')
            continue
        dx, dy = MOVES[cmd[0]]
        nx, ny = tx + dx * mm, ty + dy * mm
        if max(abs(nx - plan_x), abs(ny - plan_y)) > 30:
            print('That would be more than 30 mm from the planned spot, so the arm did not move. Tell Claude.')
            continue
    tx, ty, tz = nx, ny, nz
    arm.move_to_pos([tx, ty, tz, *img_rpy], **at_speed(3))
    print('Moved. Now', where_text(tx, ty, tz) + '.')

if released:
    print(f'Spot: x = {tx:.1f}, y = {ty:.1f}, z = {tz:.1f}   <- send this line to Claude')
    use_spot(tx, ty, tz)                  # the take-back cell uses this spot from now on
    input('Sample let go. Look at it; then press Enter (or OK) to move the arm away: '
          'up 2.5 cm, slide out, back to mid_station. ')
    go(slide_out_to_mid)
    print(f'Done. Sample {SAMPLE} is on the imaging rack. The take-back cell will fetch it from this spot.')
else:
    print('Giving up: lifting, sliding out, and putting the sample back in its own slot.')
    go(routes_for(tx, ty, tz)[2])         # up a little, slide out, high across turning back, mid_station
    arm.put_sample_back(SAMPLE)
    print(f'Sample {SAMPLE} is back in its own slot.')

### Take the sample from the imaging rack back to its own slot

In [ ]:
# ===== Take the sample from the right imaging rack back to its own slot =====
# mid_station -> high across, turning -> down beside the imaging spot -> open the gripper -> low over the rack
# -> slide in (gripper open) -> down -> close on the electrode -> up a little -> slide out, away from the camera
# -> high across, turning back -> mid_station -> into its own slot (as usual).
route = route_to_mid_from_here()
ask_to_start(f'Take sample {SAMPLE} from the right imaging rack back to its own slot. '
             f'Its own slot must be empty.', route)

print('[1] to the right imaging rack')
go(to_imaging)
arm.open_gripper()
go(slide_in)
arm.close_gripper()
opening = arm.gripper_opening   # about 9 mm when it holds an electrode, about 0 when it closes on nothing
if opening < 3:
    print(f'The gripper closed on nothing (opening {opening:.1f} mm). Going back without the sample.')
    arm.open_gripper()
    go(slide_out_to_mid)
    raise SystemExit('Nothing was taken. Check where the sample is on the imaging rack.')
print('[2] got it; out from under the camera and back to mid_station')
go(slide_out_to_mid)
print(f'[3] into its own slot')
arm.put_sample_back(SAMPLE)
print(f'Done. Sample {SAMPLE} is back in its own slot.')

### Only if you answered no at the stop above the imaging rack

Takes the held sample back to its own slot instead. It does nothing if the arm is not where the cells above stop to ask.

In [ ]:
# ===== Only if a cell above stopped above the imaging rack: put the sample back in its own slot =====
# Straight up -> across, high above the bench, turning back -> mid_station -> into its own slot (as usual).
here = arm.get_cartesian_pos()
if max(abs(here[0] - side_x), abs(here[1] - img_y), abs(here[2] - (img_z + HOVER_MM))) > 5:
    raise SystemExit('The arm is not where the cells above stop to ask, so this cell does nothing.')
if input(f'Type yes to take sample {SAMPLE} back to its own slot: ').strip().lower() != 'yes':
    raise SystemExit('Cancelled, the arm did not move.')
arm.assert_ready()
go([
    ('straight up', [side_x, img_y, SAFE_Z, *img_rpy], 30),
    ('across, turning back', [mid[0], mid[1], SAFE_Z, *mid[3:]], 30),
    ('down to mid_station', mid, 30),
])
arm.put_sample_back(SAMPLE)
print(f'Sample {SAMPLE} is back in its own slot.')